In [1]:
! which python

/home/buka2004/DRMNet/.venv10/bin/python


# Imports

In [2]:
import os
# important to make load exr
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

import math
import torch
from torch.utils.data import Dataset
import pandas as pd

from utils.file_io import load_exr, load_png, save_png  # as you specified
from torch.utils.data import DataLoader
from tqdm import tqdm

/home/buka2004/DRMNet/.venv10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Functions setup

In [15]:
class RmsePsnrInputDataset(Dataset):
    """
    Dataset that returns:
      - input image (from input_path, EXR)
      - RMSE (tensor scalar)
      - PSNR (tensor scalar, computed from CSV rmse and Imax)
    """
    def __init__(self, csv_path):
        super().__init__()
        self.df = pd.read_csv(csv_path)

        # Optional: keep only needed columns
        required_cols = ["rmse", "Imax", "input_path"]
        for c in required_cols:
            if c not in self.df.columns:
                raise ValueError(f"Missing column '{c}' in CSV")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        rmse_csv = float(row["rmse"])
        psnr_csv = float(row["psnr"])
        Imax = float(row["Imax"])
        input_path = row["input_path"]

        Imax = torch.tensor(Imax, dtype=torch.float32)
        N = torch.tensor(9, dtype=torch.float32)

        rmse = torch.tensor(rmse_csv, dtype=torch.float32)
        psnr = torch.tensor(psnr_csv, dtype=torch.float32)

        input_img = load_exr(input_path, as_torch=True) / Imax  # shape [H, W, C] or [C, H, W] depending on your utils

        sample = {
            "input": input_img,
            "rmse": rmse,
            "psnr": psnr,
            'Imax': Imax,
            'N': N,
            "input_path": input_path,
        }
        return sample


In [16]:
data_path = '/home/buka2004/DRMNet/validation_outputs/validation_results.csv'
dataset = RmsePsnrInputDataset(data_path)  # your CSV
loader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=1)

In [17]:
batch = next(iter(loader))

In [18]:
inp = batch['input'].permute(0,3,1,2)

In [19]:
inp.shape

torch.Size([8, 3, 512, 512])

# Model

(tensor(1.), tensor(1.1670e-07), tensor(0.0737))